In [1]:
import pymysql
from configparser import ConfigParser

config = ConfigParser()
config.read('../Chapter1/config.ini') # 指定設定檔的檔案路徑

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
)

print(connection.open)

True


建立資料庫

In [2]:
database = "chapter2"
with connection.cursor() as cursor:
    sql = f"""
        CREATE DATABASE IF NOT EXISTS {database}
    """
    # 執行建立的 SQL 語句
    cursor.execute(sql)
    # 執行查看資料庫
    cursor.execute("SHOW DATABASES;")
    dbs = cursor.fetchall()
print(dbs)


[{'Database': 'chapter2'}, {'Database': 'classicmodels'}, {'Database': 'information_schema'}, {'Database': 'my_databases'}, {'Database': 'my_titanic'}, {'Database': 'my_train_titanic'}, {'Database': 'mysql'}, {'Database': 'performance_schema'}, {'Database': 'sakila'}, {'Database': 'social_media_app'}, {'Database': 'sys'}, {'Database': 'transaction_test'}, {'Database': 'world'}]


建立資料表

In [3]:
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database=database,
)

""" 建立 user 資料表
    id 主鍵
    name 字串 不能為空
    age 整數 
    username 字串 不能為空 必須唯一
    password 字串 不能為空
"""
with connection.cursor() as cursor:
    sql = """
        CREATE TABLE IF NOT EXISTS user(
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(255) NOT NULL,
            age INT,
            username VARCHAR(255) NOT NULL UNIQUE,
            password VARCHAR(255) NOT NULL
            )

    """
    cursor.execute(sql)
    cursor.execute("SHOW TABLES;")
    Tables = cursor.fetchall()
    print(Tables)


[{'Tables_in_chapter2': 'user'}]


寫入資料

In [5]:
from pprint import pprint
with connection.cursor() as cursor:
    sql = """
        INSERT INTO user (name, age, username, password)
        VALUES ("Arku", 25, "Arku", "123456")
    """
    # 執行寫入的 SQL 語句
    cursor.execute(sql)
    
    # 執行查詢資料表
    cursor.execute("SELECT * FROM user;")
    # 取得查詢的所有資料
    users = cursor.fetchall()
    print(users)


IntegrityError: (1062, "Duplicate entry 'Arku' for key 'user.username'")

使用另一個連線查詢資料庫

In [6]:
connection2 = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database=database,
)
with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM user;")
    result = cursor.fetchall()

pprint(result)

[{'age': 25, 'id': 1, 'name': 'Arku', 'password': '123456', 'username': 'Arku'}]


提交資料庫變更 `conncection.commit()`

In [5]:
from pprint import pprint
with connection.cursor() as cursor:
    sql = """
        INSERT INTO user (name, age, username, password)
        VALUES ("Arku", 25, "Arku_1", "123456")
    """
    # 這邊多增加 Arku_1 的譯筆
    # 執行寫入的 SQL 語句
    # cursor.execute(sql) 這句註解掉是因為上面已經執行過了
    
    # 執行查詢資料表
    cursor.execute("SELECT * FROM user;")
    # 取得查詢的所有資料
    users = cursor.fetchall()
    print(users)
# 提交資料庫的變更
connection.commit()
    
    

[{'id': 1, 'name': 'Arku', 'age': 28, 'username': 'Arku', 'password': '123456'}]


更新資料

In [7]:
from pprint import pprint

with connection.cursor() as cursor:
    sql = """
        UPDATE user SET age=27 WHERE  username = 'Arku_1'
    """
    # 這編寫 age=20 就會全改
    cursor.execute(sql)
    
    # 可以搭配 rowcount 屬性來檢查受影響的行數
    print(cursor.rowcount)
    if cursor.rowcount == 1: # 一般來說, 因為想改 1筆, 
        # 提交資料庫的變更
        connection.commit()
    else:
        connection.rollback() # 如果不是 1筆, 就要回溯
    
    cursor.execute("SELECT * FROM user;")
    result = cursor.fetchall()

pprint(result)

0
()


刪除資料表

In [9]:
with connection.cursor() as cursor:
    sql = """
        DROP TABLE IF EXISTS user;
    """
    cursor.execute(sql)
    cursor.execute("SHOW TABLES;")
    result = cursor.fetchall()

print(result)

()


In [ ]:
import pymysql
from configparser import ConfigParser

config = ConfigParser()
config.read('../Chapter1/config.ini') # 指定設定檔的檔案路徑

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database="chapter2"
)

# 建立使用者
# 第一種, 老師示範 不建議的寫法
# f-str 不建議搭配 SQL 語句
def create_user(name, age, username, password):
    with connection.cursor() as cursor:
        sql = f"""
            INSERT INTO user (name, age, username, password)
            VALUES('{name}',{age}, '{username}', '{password}')
        """
        cursor.execute(sql)

    connection.commit()
    
    print(f"\033[31m成功寫入使用者: {username}")

create_user("Jerry", 35, "Jerry_03", "123456")

成功寫入使用者: Jerry_03


In [ ]:
# \033[31m - 37m 改顏色
# \033[1m 粗體
# \033[0m 重置

# msg = "改變顏色"
print(f"\033[32m\033[1m{msg}")

print(f"\033[32m\033[1m{msg}\033[0m")
print("重置")
# 老師常用的改變 ouput 顏色 或 字體

# 記得要再重置, 否則後面都會是一樣

# 可以去查 ANSI

改變顏色
改變顏色
重置


In [ ]:
# 課本 p26 
def check_user_password(username, password):
    with connection.cursor() as cursor:
        sql = f"""
            SELECT * FROM user WHERE username = '{username}' and password = "{password}"
            VALUES('{name}',{age}, '{username}', '{password}')
        """
        cursor.execute(sql)

    connection.commit()
    
    print(f"\033[31m成功寫入使用者: {username}")

In [ ]:
import pymysql
from configparser import ConfigParser

config = ConfigParser()
config.read('../Chapter1/config.ini') # 指定設定檔的檔案路徑

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database="chapter2"
)

# 建立使用者
# 第二種
# 習慣用佔位符 %s 的寫法
def create_user(name, age, username, password):
    with connection.cursor() as cursor:
        sql = """
            INSERT INTO user (name, age, username, password)
            VALUES(%s,%s, %s, %s)
        """
        cursor.execute(sql, (name, age, username, password))

    connection.commit()
    
    print(f"\033[31m成功寫入使用者: {username}")

create_user("Jerry", 35, "Jerry_03", "123456")

1 .正常連線, 連 SQL work bench 是用 SSL 
2. 在虛擬機, 左下角點開 , 然後安裝SSH
建 公私鑰 ssh-keygen -t rsa -f id_rsa_mca -C username -b 2048
cmd > cd .ssh
scp <file> <user>@<IP>:<path>